# ATML PA3 - Task 2: Vision-Language Models and Visual Grounding
## Kaggle Execution Notebook

This notebook runs the complete end-to-end pipeline for **Task 2** of EE-5102/CS-6304 (ATML PA3):
1. **Architecture**:
   - Vision Encoder: `google/siglip-base-patch16-224` (Always frozen, 196 patch tokens)
   - Connector: 2-layer MLP `Linear(768 -> 1024) -> GELU -> Linear(1024 -> 1024)`
   - Language Backbone: `Qwen/Qwen3-0.6B` (Original weights frozen throughout)
2. **Three-Stage Curriculum Training (4,500 total updates, batch size 16)**:
   - **Stage 1 (Updates 1 - 2,250)**: Caption alignment (Connector only, peak LR $10^{-3}$)
   - **Stage 2 (Updates 2,251 - 3,750)**: Visual instruction tuning (Connector LR $10^{-4}$ + LoRA $2\times 10^{-5}$ on $q, k, v, o$)
   - **Stage 3 (Updates 3,751 - 4,500)**: Natural VQA specialization (Connector LR $5\times 10^{-5}$ + LoRA $10^{-5}$)
3. **Development Evaluation (every 500 updates & endpoints)**:
   - Dev caption loss (96 examples)
   - Grounding dev & Natural VQA dev NLL under Correct, Complementary Swap, and No-Image inputs
   - Image-swap penalty $\Delta\text{NLL} = \text{NLL}(y | q, I_{\text{partner}}) - \text{NLL}(y | q, I_{\text{correct}})$
4. **Final Test Evaluation (Stage 3 completed model)**:
   - Grounding test (64 pairs / 128 questions) & Natural VQAv2 test (96 pairs / 192 questions)
   - 4 image conditions: Correct, Complementary Swap, Mismatched, No-Image
   - 95% bootstrap confidence intervals (4,000 resamples of complete pairs)
5. **Estimated Total Runtime**: $\approx 20\text{ to }30\text{ minutes}$ (runs comfortably in a single session!).

### Step 0: Hardware Verification & Repository Setup

In [ ]:
# Verify GPU
!nvidia-smi

import os
import sys

# Prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/ahmad-sarfraz345/ATML-PA3.git"
REPO_DIR = "/kaggle/working/ATML-PA3"

if os.path.exists("scripts/run_task2.py"):
    print("Already inside ATML-PA3 repository directory.")
    !git fetch origin && git reset --hard origin/main
elif os.path.exists(REPO_DIR):
    print("Updating existing repository in /kaggle/working/ATML-PA3...")
    %cd {REPO_DIR}
    !git fetch origin && git reset --hard origin/main
else:
    print("Cloning repository into /kaggle/working/ATML-PA3...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

### Step 1: Install Environment Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
print("Dependencies successfully installed.")

### Step 2: Stage Controls & Continuous Archiving Helper

In [ ]:
import shutil
from pathlib import Path
import zipfile

def auto_package_task2_outputs(tag="latest"):
    """Packages lightweight results archive (< 10 MB) excluding heavy .pt checkpoints."""
    source_dir = Path("outputs/task2")
    if not source_dir.exists() or not any(source_dir.iterdir()):
        return
    zip_path = Path("/kaggle/working/task2_results.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in source_dir.rglob("*"):
            if f.is_file() and f.suffix not in [".pt", ".bin", ".pth"]:
                zf.write(f, f.relative_to(source_dir))
    size_mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"[Archive] Synced {zip_path} (tag: {tag}, {size_mb:.2f} MB)")

# Execution stage controls
STAGE_PREPARE_DATA      = True
STAGE_EXTRACT_FEATURES  = True
STAGE_TRAIN_CURRICULUM  = True
STAGE_EVAL_FINAL        = True
STAGE_ANALYSIS          = True

MODEL_ID = "Qwen/Qwen3-0.6B"
SIGLIP_ID = "google/siglip-base-patch16-224"
SEED = 6304

### Step 3: Prepare Data, Form Complementary Pairs & Download Selected Images

In [ ]:
if STAGE_PREPARE_DATA:
    print("--- Executing Stage: Prepare Data ---")
    !python -u -m scripts.run_task2 --stage prepare_data --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_task2_outputs("data")

### Step 4: Extract & Cache SigLIP Patch Features (14x14 = 196 Tokens)

In [ ]:
if STAGE_EXTRACT_FEATURES:
    print("--- Executing Stage: Extract SigLIP Features ---")
    !python -u -m scripts.run_task2 --stage extract_features --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_task2_outputs("features")

### Step 5: Execute 3-Stage Curriculum Training (4,500 Updates)

In [ ]:
if STAGE_TRAIN_CURRICULUM:
    print("--- Executing Stage: 3-Stage Curriculum Training ---")
    !python -u -m scripts.run_task2 --stage train_curriculum --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_task2_outputs("curriculum")

### Step 6: Final Test Evaluations (4 Image Conditions x 2 Suites)

In [ ]:
if STAGE_EVAL_FINAL:
    print("--- Executing Stage: Final Test Evaluations ---")
    !python -u -m scripts.run_task2 --stage eval_final --model_name_or_path {MODEL_ID} --seed {SEED}
    auto_package_task2_outputs("eval_final")

### Step 7: Synthesize Evidence, Plots & Final Results Tables

In [ ]:
if STAGE_ANALYSIS:
    print("--- Executing Stage: Analysis & Plots ---")
    !python -u -m scripts.run_task2 --stage analysis --model_name_or_path {MODEL_ID} --seed {SEED}

from IPython.display import Image, display, Markdown

# Display Learning Curves
if os.path.exists("outputs/task2/learning_curves.png"):
    display(Image("outputs/task2/learning_curves.png"))

# Display Final Results Summary
if os.path.exists("outputs/task2/final_results_summary.md"):
    with open("outputs/task2/final_results_summary.md", "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

### Step 8: Final Results Package & Download

In [ ]:
source_dir = Path("outputs/task2")
zip_results_path = Path("/kaggle/working/task2_results.zip")

if source_dir.exists():
    print("Packaging lightweight Task 2 results archive...")
    with zipfile.ZipFile(zip_results_path, "w", zipfile.ZIP_DEFLATED) as zf:
        count = 0
        for f in source_dir.rglob("*"):
            if f.is_file() and f.suffix not in [".pt", ".bin", ".pth"]:
                zf.write(f, f.relative_to(source_dir))
                count += 1
    size_mb = zip_results_path.stat().st_size / (1024 * 1024)
    print(f"SUCCESS: Packaged {count} results files into {zip_results_path} ({size_mb:.2f} MB)")
    print("--> You can download task2_results.zip in seconds directly from the Kaggle Output pane!")
else:
    print(f"Warning: No files found in {source_dir} to package.")